# Safe Continual RL For Mountain Car

This notebook demonstrates a safe continual-RL pipeline for `TunableMountainCar-v0`.

The pipeline is:

1. Train a source PPO policy on the default dynamics.
2. Certify a border-safety property with IBP and CROWN-style bounds.
3. Define downstream task shifts such as heavier gravity, weaker force, and shifted goals.
4. Adapt the source policy with three constrained methods:
   - Rashomon parameter bounds plus PPO projected gradient descent.
   - Lagrangian PPO with a learned safety multiplier.
   - CPO-style trust-region PPO with a safety surrogate and KL guard.
5. Re-certify every adapted policy before accepting it.

The notebook is intentionally self-contained. It uses project utilities where they already exist and keeps experiment-specific glue in notebook cells.

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import TensorDataset
from stable_baselines3 import PPO

try:
    from IPython.display import display
except Exception:  # pragma: no cover - fallback for script smoke tests
    def display(obj: Any) -> None:
        print(obj)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "experiments/pipelines/mountaincar").exists() and (candidate / "core").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the CertifiedContinualLearning repository.")


PROJECT_ROOT = find_project_root()
CORE_ROOT = PROJECT_ROOT / "core"
for path in (PROJECT_ROOT, CORE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from abstract_gradient_training.bounded_models import CROWNBoundedModel, IntervalBoundedModel
from experiments.pipelines.mountaincar.core.env.env_factory import make_mountaincar_env
from experiments.pipelines.mountaincar.core.methods.source_train import build_actor_critic
from experiments.utils.ppo_utils import PPOConfig, evaluate_with_success, ppo_train
from src.trainer.IntervalTrainer import IntervalTrainer

print(f"Project root: {PROJECT_ROOT}")

## Configuration

`FAST_DEMO=True` keeps the notebook suitable for interactive development. For a stronger experiment, set `FAST_DEMO=False`, increase the PPO timesteps, and use more Rashomon iterations.

In [10]:
FAST_DEMO = False
RUN_FULL_PIPELINE = True
RUN_ALL_DOWNSTREAM_TASKS = False
SOURCE_READY = False


@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 0
    device: str = "cpu"
    hidden_size: int = 64
    append_task_id: bool = False
    task_id: float = 0.0
    solved_reward_threshold: float = -110.0
    source_timesteps: int = 60_000 if FAST_DEMO else 250_000
    adapt_timesteps: int = 20_000 if FAST_DEMO else 120_000
    rollout_steps: int = 512 if FAST_DEMO else 2048
    update_epochs: int = 4 if FAST_DEMO else 10
    minibatch_size: int = 64
    eval_episodes: int = 10 if FAST_DEMO else 100
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_coef: float = 0.2
    ent_coef: float = 0.0
    vf_coef: float = 0.5
    lr: float = 3e-4
    max_grad_norm: float = 0.5
    safety_border_width: float = 0.20
    safety_grid_positions: int = 21 if FAST_DEMO else 41
    safety_grid_velocities: int = 21 if FAST_DEMO else 41
    safety_samples: int = 2_000 if FAST_DEMO else 20_000
    rashomon_iters: int = 300 if FAST_DEMO else 5_000
    rashomon_checkpoint: int = 100 if FAST_DEMO else 500
    rashomon_inverse_temp_start: int = 1
    rashomon_inverse_temp_max: int = 200
    cost_limit: float = 0.01
    lagrangian_lr: float = 0.05
    initial_lagrange_multiplier: float = 1.0
    cpo_target_kl: float = 0.015
    cpo_safety_penalty: float = 25.0


CFG = ExperimentConfig()
ARTIFACT_ROOT = PROJECT_ROOT / "experiments/pipelines/mountaincar/artifacts/safe_continual_rl"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything(CFG.seed)
print(CFG)

ExperimentConfig(seed=0, device='cpu', hidden_size=64, append_task_id=False, task_id=0.0, solved_reward_threshold=-110.0, source_timesteps=250000, adapt_timesteps=120000, rollout_steps=2048, update_epochs=10, minibatch_size=64, eval_episodes=100, gamma=0.99, gae_lambda=0.95, clip_coef=0.2, ent_coef=0.0, vf_coef=0.5, lr=0.0003, max_grad_norm=0.5, safety_border_width=0.2, safety_grid_positions=41, safety_grid_velocities=41, safety_samples=20000, rashomon_iters=5000, rashomon_checkpoint=500, rashomon_inverse_temp_start=1, rashomon_inverse_temp_max=200, cost_limit=0.01, lagrangian_lr=0.05, initial_lagrange_multiplier=1.0, cpo_target_kl=0.015, cpo_safety_penalty=25.0)


## Safety Spec And Downstream Tasks

The default safety-critical region is the left border of the Mountain Car track. In that box, the deterministic policy must select action `2`, `push_right`, so that it pushes away from the border.

In [11]:
ACTION_NAMES = {0: "push_left", 1: "no_push", 2: "push_right"}


@dataclass(frozen=True)
class SafetyBox:
    name: str
    lower: tuple[float, ...]
    upper: tuple[float, ...]
    target_action: int


@dataclass(frozen=True)
class DownstreamTask:
    name: str
    env_kwargs: dict[str, float]
    description: str


DEFAULT_TASK = DownstreamTask(
    name="default",
    env_kwargs={},
    description="Default Gymnasium Mountain Car dynamics.",
)

DOWNSTREAM_TASKS = {
    "heavy_gravity": DownstreamTask(
        name="heavy_gravity",
        env_kwargs={"gravity": 0.0035},
        description="Gravity is stronger, so momentum management is harder.",
    ),
    "weak_force": DownstreamTask(
        name="weak_force",
        env_kwargs={"force": 0.00075},
        description="The engine is weaker, changing the acceleration dynamics.",
    ),
    "hard_goal": DownstreamTask(
        name="hard_goal",
        env_kwargs={"goal_position": 0.55},
        description="The goal is moved closer to the right edge.",
    ),
    "wide_track": DownstreamTask(
        name="wide_track",
        env_kwargs={"min_position": -1.3, "max_position": 0.7, "goal_position": 0.6},
        description="The board is wider and the goal moves with the right side.",
    ),
}

SELECTED_TASK_NAMES = list(DOWNSTREAM_TASKS) if RUN_ALL_DOWNSTREAM_TASKS else ["heavy_gravity"]


class BorderSafetySpec:
    def __init__(self, border_width: float = 0.20, include_right_border: bool = False):
        self.border_width = float(border_width)
        self.include_right_border = bool(include_right_border)

    def required_action(self, obs: np.ndarray, base_env: Any) -> int | None:
        position = float(np.asarray(obs)[0])
        min_position = float(base_env.min_position)
        max_position = float(base_env.max_position)
        goal_position = float(base_env.goal_position)
        if position <= min_position + self.border_width:
            return 2
        if self.include_right_border and position >= max_position - self.border_width and position < goal_position:
            return 0
        return None

    def boxes(self, env: gym.Env, *, append_task_id: bool, task_id: float) -> list[SafetyBox]:
        base = env.unwrapped
        lower = [float(base.min_position), -float(base.max_speed)]
        upper = [float(base.min_position + self.border_width), float(base.max_speed)]
        if append_task_id:
            lower.append(float(task_id))
            upper.append(float(task_id))
        boxes = [
            SafetyBox(
                name="left_border_push_right",
                lower=tuple(lower),
                upper=tuple(upper),
                target_action=2,
            )
        ]
        if self.include_right_border:
            right_low = float(base.max_position - self.border_width)
            if right_low < float(base.goal_position):
                lower = [right_low, -float(base.max_speed)]
                upper = [float(base.max_position), float(base.max_speed)]
                if append_task_id:
                    lower.append(float(task_id))
                    upper.append(float(task_id))
                boxes.append(
                    SafetyBox(
                        name="right_border_push_left",
                        lower=tuple(lower),
                        upper=tuple(upper),
                        target_action=0,
                    )
                )
        return boxes


SAFETY_SPEC = BorderSafetySpec(border_width=CFG.safety_border_width)
print("Downstream task options:")
for task in DOWNSTREAM_TASKS.values():
    print(f"- {task.name}: {task.env_kwargs} | {task.description}")

Downstream task options:
- heavy_gravity: {'gravity': 0.0035} | Gravity is stronger, so momentum management is harder.
- weak_force: {'force': 0.00075} | The engine is weaker, changing the acceleration dynamics.
- hard_goal: {'goal_position': 0.55} | The goal is moved closer to the right edge.
- wide_track: {'min_position': -1.3, 'max_position': 0.7, 'goal_position': 0.6} | The board is wider and the goal moves with the right side.


## Environment Helpers

Training uses potential-based reward shaping to get a usable source policy. Evaluation and certification use the original environment reward and the explicit safety property.

In [12]:
class MountainCarShapedReward(gym.Wrapper):
    def __init__(self, env: gym.Env, gamma: float = 0.99):
        super().__init__(env)
        self.gamma = float(gamma)
        self.previous_potential = 0.0

    @staticmethod
    def potential(obs: np.ndarray) -> float:
        position = float(obs[0])
        velocity = float(obs[1])
        return 10.0 * position + 100.0 * abs(velocity)

    def reset(self, **kwargs: Any) -> tuple[np.ndarray, dict[str, Any]]:
        obs, info = self.env.reset(**kwargs)
        self.previous_potential = self.potential(obs)
        return obs, info

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict[str, Any]]:
        obs, reward, terminated, truncated, info = self.env.step(action)
        current_potential = self.potential(obs)
        shaped = float(reward + self.gamma * current_potential - self.previous_potential)
        self.previous_potential = current_potential
        if float(obs[0]) >= float(self.unwrapped.goal_position):
            shaped += 100.0
        return obs, shaped, terminated, truncated, info


class SafetyCostWrapper(gym.Wrapper):
    def __init__(self, env: gym.Env, spec: BorderSafetySpec):
        super().__init__(env)
        self.safety_spec = spec
        self._last_obs: np.ndarray | None = None

    def reset(self, **kwargs: Any) -> tuple[np.ndarray, dict[str, Any]]:
        obs, info = self.env.reset(**kwargs)
        self._last_obs = np.asarray(obs, dtype=np.float32).copy()
        return obs, info

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict[str, Any]]:
        if self._last_obs is None:
            raise RuntimeError("SafetyCostWrapper.step called before reset.")
        required = self.safety_spec.required_action(self._last_obs, self.unwrapped)
        violation = bool(required is not None and int(action) != int(required))
        obs, reward, terminated, truncated, info = self.env.step(int(action))
        step_info = dict(info)
        step_info["cost"] = float(violation)
        step_info["safe"] = not violation
        step_info["required_safe_action"] = required
        self._last_obs = np.asarray(obs, dtype=np.float32).copy()
        return obs, reward, terminated, truncated, step_info


def make_task_env(
    task: DownstreamTask,
    *,
    seed: int | None = None,
    shaped: bool = False,
    safety_cost: bool = False,
    render_mode: str | None = None,
) -> gym.Env:
    env = make_mountaincar_env(
        append_task_id=CFG.append_task_id,
        task_id=CFG.task_id,
        render_mode=render_mode,
        **task.env_kwargs,
    )
    if safety_cost:
        env = SafetyCostWrapper(env, SAFETY_SPEC)
    if shaped:
        env = MountainCarShapedReward(env, gamma=CFG.gamma)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)
    return env


def make_ppo_config(total_timesteps: int, *, seed: int) -> PPOConfig:
    return PPOConfig(
        seed=seed,
        total_timesteps=int(total_timesteps),
        eval_episodes=CFG.eval_episodes,
        rollout_steps=CFG.rollout_steps,
        update_epochs=CFG.update_epochs,
        minibatch_size=CFG.minibatch_size,
        gamma=CFG.gamma,
        gae_lambda=CFG.gae_lambda,
        clip_coef=CFG.clip_coef,
        ent_coef=CFG.ent_coef,
        vf_coef=CFG.vf_coef,
        lr=CFG.lr,
        max_grad_norm=CFG.max_grad_norm,
        device=CFG.device,
        early_stop_reward_threshold=CFG.solved_reward_threshold,
        early_stop_failure_rate_threshold=0.0,
    )

## Policy Training And Evaluation

In [ ]:
SOURCE_POLICY_PATH = PROJECT_ROOT / "experiments/pipelines/mountaincar/artifacts/seed_0/best_model.zip"


def build_actor_critic_for_env(env: gym.Env) -> tuple[nn.Sequential, nn.Sequential]:
    if not isinstance(env.observation_space, gym.spaces.Box):
        raise ValueError("Expected Box observation space.")
    if not isinstance(env.action_space, gym.spaces.Discrete):
        raise ValueError("Expected Discrete action space.")
    obs_dim = int(env.observation_space.shape[0])
    n_actions = int(env.action_space.n)
    return build_actor_critic(obs_dim=obs_dim, n_actions=n_actions, hidden_size=CFG.hidden_size)


def is_sb3_policy(policy: Any) -> bool:
    return hasattr(policy, "predict") and hasattr(policy, "policy")


def extract_policy_network(policy: Any) -> nn.Sequential:
    """Return a PyTorch actor network for verification/adaptation.

    SB3 PPO stores the actor as `policy_net -> action_net`; the loaded object is
    kept as `source_actor`, but this helper gives notebook methods the network
    they need when they operate directly on logits.
    """
    if isinstance(policy, nn.Sequential):
        return copy.deepcopy(policy)
    if is_sb3_policy(policy):
        modules = list(policy.policy.mlp_extractor.policy_net.children()) + [policy.policy.action_net]
        return copy.deepcopy(nn.Sequential(*modules)).cpu().eval()
    raise TypeError(f"Unsupported policy type: {type(policy)!r}")


def policy_supports_interval_trainer(policy: Any) -> bool:
    supported = (nn.Linear, nn.ReLU, nn.Flatten, nn.Dropout)
    try:
        network = extract_policy_network(policy)
    except TypeError:
        return False
    return all(isinstance(module, supported) for module in network)


def deterministic_action(policy: Any, obs: np.ndarray, *, device: str = CFG.device) -> int:
    if is_sb3_policy(policy):
        action, _ = policy.predict(np.asarray(obs, dtype=np.float32), deterministic=True)
        return int(np.asarray(action).item())

    actor = extract_policy_network(policy).to(device).eval()
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = actor(obs_t)
    return int(torch.argmax(logits, dim=-1).item())


def evaluate_actor_on_task(
    policy: Any,
    task: DownstreamTask,
    *,
    episodes: int | None = None,
    seed: int = CFG.seed,
    safety_cost: bool = True,
) -> dict[str, float]:
    env = make_task_env(task, seed=seed, shaped=False, safety_cost=safety_cost)
    n_episodes = int(episodes or CFG.eval_episodes)
    returns: list[float] = []
    failures = 0
    successes = 0

    try:
        goal_position = float(env.unwrapped.goal_position)
        for episode in range(n_episodes):
            obs, _ = env.reset(seed=seed + episode)
            episode_return = 0.0
            episode_failed = False
            episode_success = False
            done = False
            while not done:
                action = deterministic_action(policy, obs)
                obs, reward, terminated, truncated, info = env.step(action)
                episode_return += float(reward)
                is_safe = info.get("safe", None)
                if is_safe is None:
                    is_safe = float(info.get("cost", 0.0)) == 0.0
                episode_failed = episode_failed or (not bool(is_safe))
                episode_success = episode_success or bool(info.get("is_success", False)) or float(obs[0]) >= goal_position
                done = bool(terminated or truncated)
            returns.append(episode_return)
            failures += int(episode_failed)
            successes += int(episode_success)
    finally:
        env.close()

    returns_array = np.asarray(returns, dtype=np.float64)
    return {
        "mean_reward": float(returns_array.mean()),
        "std_reward": float(returns_array.std()),
        "failure_rate": float(failures / n_episodes),
        "success_rate": float(successes / n_episodes),
    }


def load_source_policy() -> tuple[Any, None, None, dict[str, float]]:
    if not SOURCE_POLICY_PATH.exists():
        raise FileNotFoundError(f"Missing source policy checkpoint: {SOURCE_POLICY_PATH}")
    source_actor = PPO.load(str(SOURCE_POLICY_PATH), device=CFG.device)
    source_critic = None
    source_training_data = None
    source_metrics = evaluate_actor_on_task(
        source_actor,
        DEFAULT_TASK,
        episodes=CFG.eval_episodes,
        seed=CFG.seed + 20_000,
        safety_cost=True,
    )
    return source_actor, source_critic, source_training_data, source_metrics


if RUN_FULL_PIPELINE:
    source_actor, source_critic, source_training_data, source_metrics = load_source_policy()
    print(f"Loaded source PPO checkpoint: {SOURCE_POLICY_PATH}")
    print("Source metrics:", source_metrics)
else:
    source_actor = source_critic = source_training_data = source_metrics = None
    print("Set RUN_FULL_PIPELINE=True to load the source policy.")

In [17]:
import stable_baselines3 as sb3
from stable_baselines3 import PPO

In [ ]:
# The source policy is loaded in the previous cell via `load_source_policy()`:
# source_actor = PPO.load('/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning/experiments/pipelines/mountaincar/artifacts/seed_0/best_model.zip')
# source_critic = None

## IBP And CROWN Verification

For a safety box, the certificate is positive when the target action lower-bound logit exceeds every competing action upper-bound logit. The loaded SB3 checkpoint uses `Tanh`, so the notebook uses manual IBP for that policy and CROWN only for supported ReLU actors.


In [ ]:
VerificationMethod = Literal["ibp", "crown", "alpha_crown"]


def affine_interval(
    weight: np.ndarray,
    bias: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    weight_pos = np.maximum(weight, 0.0)
    weight_neg = np.minimum(weight, 0.0)
    next_lower = weight_pos @ lower + weight_neg @ upper + bias
    next_upper = weight_pos @ upper + weight_neg @ lower + bias
    return next_lower, next_upper


def manual_interval_forward(policy: Any, input_lower: np.ndarray, input_upper: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    modules = list(extract_policy_network(policy).children())
    lower = input_lower.astype(np.float64)
    upper = input_upper.astype(np.float64)
    for module in modules:
        if isinstance(module, nn.Linear):
            lower, upper = affine_interval(
                module.weight.detach().cpu().numpy().astype(np.float64),
                module.bias.detach().cpu().numpy().astype(np.float64),
                lower,
                upper,
            )
        elif isinstance(module, nn.ReLU):
            lower, upper = np.maximum(lower, 0.0), np.maximum(upper, 0.0)
        elif isinstance(module, nn.Tanh):
            lower, upper = np.tanh(lower), np.tanh(upper)
        elif isinstance(module, nn.Flatten):
            lower, upper = lower.reshape(-1), upper.reshape(-1)
        else:
            raise TypeError(f"Unsupported module for manual IBP: {module!r}")
    return lower, upper


def verification_methods_for_policy(policy: Any) -> tuple[VerificationMethod, ...]:
    if policy_supports_interval_trainer(policy):
        return ("ibp", "crown", "alpha_crown")
    print(
        "Policy contains modules outside the repo CROWN verifier support "
        "(for the loaded SB3 checkpoint this is usually Tanh), so using IBP only."
    )
    return ("ibp",)


def _bounded_model(actor: Any, method: VerificationMethod):
    actor_cpu = extract_policy_network(actor).cpu().eval()
    if method == "ibp":
        return IntervalBoundedModel(actor_cpu, trainable=False)
    if method == "crown":
        return CROWNBoundedModel(actor_cpu, trainable=False, relu_relaxation="parallel")
    if method == "alpha_crown":
        return CROWNBoundedModel(
            actor_cpu,
            trainable=False,
            relu_relaxation="optimizable",
            alpha_crown_iters=20 if FAST_DEMO else 100,
            alpha_crown_lr=0.1,
            optimize_inter=False,
        )
    raise ValueError(f"Unknown verification method: {method}")


def certify_action_box(
    actor: Any,
    box: SafetyBox,
    *,
    method: VerificationMethod,
) -> dict[str, Any]:
    if method == "ibp" and not policy_supports_interval_trainer(actor):
        lb, ub = manual_interval_forward(
            actor,
            np.asarray(box.lower, dtype=np.float64),
            np.asarray(box.upper, dtype=np.float64),
        )
    else:
        lower_t = torch.tensor(box.lower, dtype=torch.float32).unsqueeze(0)
        upper_t = torch.tensor(box.upper, dtype=torch.float32).unsqueeze(0)
        verifier = _bounded_model(actor, method)
        # Alpha-CROWN optimizes relaxation parameters internally, so this call must
        # not be wrapped in torch.no_grad().
        output_lower, output_upper = verifier.bound_forward(lower_t, upper_t)
        lb = output_lower.squeeze(0).detach().cpu().numpy()
        ub = output_upper.squeeze(0).detach().cpu().numpy()

    competitors = [a for a in range(lb.shape[0]) if a != box.target_action]
    target_lower = float(lb[box.target_action])
    best_competing_upper = float(max(ub[a] for a in competitors))
    margin = target_lower - best_competing_upper
    return {
        "box": box.name,
        "method": method,
        "target_action": int(box.target_action),
        "target_action_name": ACTION_NAMES[int(box.target_action)],
        "certified": bool(margin > 0.0),
        "margin": float(margin),
        "target_lower": target_lower,
        "best_competing_upper": best_competing_upper,
        "output_lower": lb.tolist(),
        "output_upper": ub.tolist(),
    }


def sample_box_actions(
    actor: Any,
    box: SafetyBox,
    *,
    n: int = CFG.safety_samples,
    seed: int = CFG.seed,
) -> dict[str, Any]:
    rng = np.random.default_rng(seed)
    lower = np.asarray(box.lower, dtype=np.float32)
    upper = np.asarray(box.upper, dtype=np.float32)
    samples = rng.uniform(lower, upper, size=(n, lower.size)).astype(np.float32)
    actions = [deterministic_action(actor, sample) for sample in samples]
    counts = {ACTION_NAMES[action]: int(np.count_nonzero(np.asarray(actions) == action)) for action in ACTION_NAMES}
    return {
        "box": box.name,
        "n": int(n),
        "counts": counts,
        "all_target": bool(all(action == box.target_action for action in actions)),
    }


def verify_policy_safety(actor: Any, task: DownstreamTask, *, label: str) -> list[dict[str, Any]]:
    env = make_task_env(task, seed=CFG.seed, shaped=False, safety_cost=False)
    try:
        boxes = SAFETY_SPEC.boxes(env, append_task_id=CFG.append_task_id, task_id=CFG.task_id)
    finally:
        env.close()

    rows: list[dict[str, Any]] = []
    methods = verification_methods_for_policy(actor)
    for box in boxes:
        rows.append({"label": label, "check": "sample", **sample_box_actions(actor, box)})
        for method in methods:
            rows.append({"label": label, "check": "certificate", **certify_action_box(actor, box, method=method)})
    return rows


if RUN_FULL_PIPELINE and source_actor is not None:
    source_safety_rows = verify_policy_safety(source_actor, DEFAULT_TASK, label="source_default")
    display(pd.DataFrame(source_safety_rows))
    source_cert_rows = [row for row in source_safety_rows if row.get("check") == "certificate"]
    source_sample_rows = [row for row in source_safety_rows if row.get("check") == "sample"]
    source_performance_ok = bool(source_metrics["mean_reward"] >= CFG.solved_reward_threshold)
    source_cert_ok = bool(source_cert_rows and all(row["certified"] for row in source_cert_rows))
    source_sample_ok = bool(source_sample_rows and all(row["all_target"] for row in source_sample_rows))
    SOURCE_READY = bool(source_performance_ok and source_cert_ok and source_sample_ok)
    print(
        "Source gate | "
        f"performance_ok={source_performance_ok} | "
        f"sample_ok={source_sample_ok} | "
        f"certificate_ok={source_cert_ok} | "
        f"ready_for_adaptation={SOURCE_READY}"
    )
else:
    source_safety_rows = []
    SOURCE_READY = False

## Rashomon Safety Dataset And Bounds

The Rashomon set is computed over a finite safety dataset sampled from the critical region plus optional source rollouts. The final continuous-state safety proof still comes from IBP/CROWN verification over the full box.

In [7]:
def grid_dataset_from_boxes(boxes: list[SafetyBox], *, n_actions: int = 3) -> TensorDataset:
    observations: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    for box in boxes:
        lower = np.asarray(box.lower, dtype=np.float32)
        upper = np.asarray(box.upper, dtype=np.float32)
        positions = np.linspace(lower[0], upper[0], CFG.safety_grid_positions, dtype=np.float32)
        velocities = np.linspace(lower[1], upper[1], CFG.safety_grid_velocities, dtype=np.float32)
        for position in positions:
            for velocity in velocities:
                obs = np.array([position, velocity], dtype=np.float32)
                if CFG.append_task_id:
                    obs = np.concatenate([obs, np.array([CFG.task_id], dtype=np.float32)])
                action_mask = np.zeros(n_actions, dtype=np.float32)
                action_mask[box.target_action] = 1.0
                observations.append(obs)
                labels.append(action_mask)
    return TensorDataset(torch.tensor(np.asarray(observations)), torch.tensor(np.asarray(labels)))


def rollout_label_dataset(
    actor: nn.Sequential,
    task: DownstreamTask,
    *,
    episodes: int = 3 if FAST_DEMO else 20,
    n_actions: int = 3,
) -> TensorDataset:
    observations: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    env = make_task_env(task, seed=CFG.seed, shaped=False, safety_cost=False)
    try:
        for episode in range(episodes):
            obs, _ = env.reset(seed=CFG.seed + episode)
            done = False
            while not done:
                obs_np = np.asarray(obs, dtype=np.float32)
                action = deterministic_action(actor, obs_np)
                mask = np.zeros(n_actions, dtype=np.float32)
                mask[action] = 1.0
                observations.append(obs_np.copy())
                labels.append(mask)
                obs, _, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
    finally:
        env.close()
    return TensorDataset(torch.tensor(np.asarray(observations)), torch.tensor(np.asarray(labels)))


def concatenate_tensor_datasets(datasets: list[TensorDataset]) -> TensorDataset:
    xs = torch.cat([dataset.tensors[0] for dataset in datasets], dim=0)
    ys = torch.cat([dataset.tensors[1] for dataset in datasets], dim=0)
    return TensorDataset(xs.float(), ys.float())


def compute_surrogate_threshold(dataset: TensorDataset) -> float:
    valid_counts = dataset.tensors[1].sum(dim=1)
    max_valid = float(valid_counts.max().item())
    if max_valid <= 0:
        raise ValueError("Rashomon dataset has no valid actions.")
    return float(max_valid / (max_valid + 1.0))


def calibrate_inverse_temperature(
    actor: nn.Sequential,
    dataset: TensorDataset,
    surrogate_threshold: float,
) -> int:
    actor.eval()
    obs, action_mask = dataset.tensors
    with torch.no_grad():
        logits = actor(obs.to(CFG.device)).cpu()
    for inverse_temp in range(CFG.rashomon_inverse_temp_start, CFG.rashomon_inverse_temp_max + 1):
        probs = torch.softmax(logits * inverse_temp, dim=1)
        valid_mass = (probs * action_mask).sum(dim=1)
        if float(valid_mass.min().item()) >= surrogate_threshold:
            return int(inverse_temp)
    best = float(((torch.softmax(logits * CFG.rashomon_inverse_temp_max, dim=1) * action_mask).sum(dim=1)).min().item())
    raise RuntimeError(
        f"Could not calibrate inverse temperature. Best min valid mass={best:.6f}, threshold={surrogate_threshold:.6f}."
    )


def compute_rashomon_bounds(actor: nn.Sequential, task: DownstreamTask) -> tuple[list[torch.Tensor], list[torch.Tensor], dict[str, Any]]:
    env = make_task_env(task, seed=CFG.seed, shaped=False, safety_cost=False)
    try:
        boxes = SAFETY_SPEC.boxes(env, append_task_id=CFG.append_task_id, task_id=CFG.task_id)
    finally:
        env.close()

    safety_dataset = grid_dataset_from_boxes(boxes)
    rollout_dataset = rollout_label_dataset(actor, DEFAULT_TASK)
    rashomon_dataset = concatenate_tensor_datasets([safety_dataset, rollout_dataset])
    surrogate_threshold = compute_surrogate_threshold(rashomon_dataset)
    inverse_temp = calibrate_inverse_temperature(actor, rashomon_dataset, surrogate_threshold)

    trainer = IntervalTrainer(
        model=copy.deepcopy(actor).cpu(),
        min_soft_acc_limit=surrogate_threshold,
        min_hard_acc_limit=1.0,
        min_acc_increment=0,
        seed=CFG.seed,
        n_iters=CFG.rashomon_iters,
        T=inverse_temp,
        checkpoint=CFG.rashomon_checkpoint,
    )
    trainer.compute_rashomon_set(dataset=rashomon_dataset, multi_label=True, aggregation="min")

    cert_values = [float(cert) if cert is not None else float("-inf") for cert in trainer.certificates]
    valid_indices = [idx for idx, cert in enumerate(cert_values) if cert >= 1.0]
    if not valid_indices:
        raise RuntimeError(f"No Rashomon checkpoint certified the finite safety dataset. Certificates={cert_values}")
    selected_idx = valid_indices[-1]
    bounded_model = trainer.bounds[selected_idx]
    bounds_l = [param.detach().cpu() for param in bounded_model.param_l]
    bounds_u = [param.detach().cpu() for param in bounded_model.param_u]
    info = {
        "dataset_size": int(len(rashomon_dataset)),
        "safety_dataset_size": int(len(safety_dataset)),
        "rollout_dataset_size": int(len(rollout_dataset)),
        "surrogate_threshold": float(surrogate_threshold),
        "inverse_temp": int(inverse_temp),
        "certificates": cert_values,
        "selected_checkpoint": int(selected_idx),
    }
    return bounds_l, bounds_u, info

## Downstream Adaptation Methods

In [8]:
def adapt_with_rashomon_pgd(
    source_actor: nn.Sequential,
    source_critic: nn.Sequential,
    task: DownstreamTask,
) -> tuple[nn.Sequential, nn.Sequential, dict[str, Any]]:
    bounds_l, bounds_u, bounds_info = compute_rashomon_bounds(source_actor, task)
    train_env = make_task_env(task, seed=CFG.seed + 30_000, shaped=True, safety_cost=True)
    eval_env = make_task_env(task, seed=CFG.seed + 40_000, shaped=False, safety_cost=True)
    cfg = make_ppo_config(CFG.adapt_timesteps, seed=CFG.seed + 1)
    actor, critic, training_data = ppo_train(
        env=train_env,
        cfg=cfg,
        actor_warm_start=copy.deepcopy(source_actor),
        critic_warm_start=copy.deepcopy(source_critic),
        actor_param_bounds_l=bounds_l,
        actor_param_bounds_u=bounds_u,
        early_stop_eval_env=eval_env,
        return_training_data=True,
    )
    info = {
        "method": "rashomon_pgd",
        "rashomon": bounds_info,
        "training_steps": int(len(training_data["states"])),
    }
    return actor, critic, info


def compute_gae_1d(rewards: np.ndarray, dones: np.ndarray, values: np.ndarray, last_value: float, gamma: float, lam: float) -> tuple[np.ndarray, np.ndarray]:
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(len(rewards))):
        next_nonterminal = 1.0 - dones[t]
        next_value = last_value if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value * next_nonterminal - values[t]
        last_gae = delta + gamma * lam * next_nonterminal * last_gae
        advantages[t] = last_gae
    returns = advantages + values
    return advantages, returns


def make_value_network(obs_dim: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(obs_dim, CFG.hidden_size),
        nn.ReLU(),
        nn.Linear(CFG.hidden_size, CFG.hidden_size),
        nn.ReLU(),
        nn.Linear(CFG.hidden_size, 1),
    )


def train_constrained_ppo(
    source_actor: nn.Sequential,
    source_critic: nn.Sequential,
    task: DownstreamTask,
    *,
    method: Literal["lagrangian", "cpo_style"],
) -> tuple[nn.Sequential, nn.Sequential, dict[str, Any]]:
    env = make_task_env(task, seed=CFG.seed + 50_000, shaped=True, safety_cost=True)
    if not isinstance(env.observation_space, gym.spaces.Box) or not isinstance(env.action_space, gym.spaces.Discrete):
        raise ValueError("Constrained PPO demo expects Box observations and Discrete actions.")
    obs_dim = int(env.observation_space.shape[0])
    n_actions = int(env.action_space.n)

    actor = copy.deepcopy(source_actor).to(CFG.device)
    reward_critic = copy.deepcopy(source_critic).to(CFG.device)
    cost_critic = make_value_network(obs_dim).to(CFG.device)
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(reward_critic.parameters()) + list(cost_critic.parameters()),
        lr=CFG.lr,
    )
    lagrange_multiplier = float(CFG.initial_lagrange_multiplier)
    obs, _ = env.reset(seed=CFG.seed)
    global_step = 0
    updates = max(1, math.ceil(CFG.adapt_timesteps / CFG.rollout_steps))
    cost_history: list[float] = []
    skipped_updates = 0

    for update_idx in range(updates):
        obs_buf = np.zeros((CFG.rollout_steps, obs_dim), dtype=np.float32)
        act_buf = np.zeros((CFG.rollout_steps,), dtype=np.int64)
        logp_buf = np.zeros((CFG.rollout_steps,), dtype=np.float32)
        rew_buf = np.zeros((CFG.rollout_steps,), dtype=np.float32)
        cost_buf = np.zeros((CFG.rollout_steps,), dtype=np.float32)
        done_buf = np.zeros((CFG.rollout_steps,), dtype=np.float32)
        value_buf = np.zeros((CFG.rollout_steps,), dtype=np.float32)
        cost_value_buf = np.zeros((CFG.rollout_steps,), dtype=np.float32)

        for t in range(CFG.rollout_steps):
            obs_buf[t] = obs
            obs_t = torch.tensor(obs, dtype=torch.float32, device=CFG.device).unsqueeze(0)
            with torch.no_grad():
                logits = actor(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                action_t = dist.sample()
                logp_t = dist.log_prob(action_t)
                value_t = reward_critic(obs_t).squeeze(-1)
                cost_value_t = cost_critic(obs_t).squeeze(-1)
            action = int(action_t.item())
            next_obs, reward, terminated, truncated, info = env.step(action)
            done = bool(terminated or truncated)
            obs_buf[t] = obs
            act_buf[t] = action
            logp_buf[t] = float(logp_t.item())
            rew_buf[t] = float(reward)
            cost_buf[t] = float(info.get("cost", 0.0))
            done_buf[t] = float(done)
            value_buf[t] = float(value_t.item())
            cost_value_buf[t] = float(cost_value_t.item())
            obs = next_obs
            global_step += 1
            if done:
                obs, _ = env.reset()
            if global_step >= CFG.adapt_timesteps:
                obs_buf = obs_buf[: t + 1]
                act_buf = act_buf[: t + 1]
                logp_buf = logp_buf[: t + 1]
                rew_buf = rew_buf[: t + 1]
                cost_buf = cost_buf[: t + 1]
                done_buf = done_buf[: t + 1]
                value_buf = value_buf[: t + 1]
                cost_value_buf = cost_value_buf[: t + 1]
                break

        with torch.no_grad():
            last_obs_t = torch.tensor(obs, dtype=torch.float32, device=CFG.device).unsqueeze(0)
            last_value = float(reward_critic(last_obs_t).item())
            last_cost_value = float(cost_critic(last_obs_t).item())

        reward_adv, reward_ret = compute_gae_1d(rew_buf, done_buf, value_buf, last_value, CFG.gamma, CFG.gae_lambda)
        cost_adv, cost_ret = compute_gae_1d(cost_buf, done_buf, cost_value_buf, last_cost_value, CFG.gamma, CFG.gae_lambda)
        reward_adv = (reward_adv - reward_adv.mean()) / (reward_adv.std() + 1e-8)
        cost_adv = (cost_adv - cost_adv.mean()) / (cost_adv.std() + 1e-8)
        mean_cost = float(cost_buf.mean()) if len(cost_buf) else 0.0
        cost_history.append(mean_cost)

        obs_t = torch.tensor(obs_buf, dtype=torch.float32, device=CFG.device)
        act_t = torch.tensor(act_buf, dtype=torch.int64, device=CFG.device)
        old_logp_t = torch.tensor(logp_buf, dtype=torch.float32, device=CFG.device)
        reward_adv_t = torch.tensor(reward_adv, dtype=torch.float32, device=CFG.device)
        cost_adv_t = torch.tensor(cost_adv, dtype=torch.float32, device=CFG.device)
        reward_ret_t = torch.tensor(reward_ret, dtype=torch.float32, device=CFG.device)
        cost_ret_t = torch.tensor(cost_ret, dtype=torch.float32, device=CFG.device)

        batch_size = len(obs_t)
        indices = np.arange(batch_size)
        for _ in range(CFG.update_epochs):
            np.random.shuffle(indices)
            for start in range(0, batch_size, CFG.minibatch_size):
                mb_idx = indices[start : start + CFG.minibatch_size]
                logits = actor(obs_t[mb_idx])
                dist = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(act_t[mb_idx])
                entropy = dist.entropy().mean()
                ratio = torch.exp(new_logp - old_logp_t[mb_idx])
                approx_kl = torch.mean(old_logp_t[mb_idx] - new_logp).detach()

                if method == "lagrangian":
                    combined_adv = reward_adv_t[mb_idx] - lagrange_multiplier * cost_adv_t[mb_idx]
                    pg1 = -combined_adv * ratio
                    pg2 = -combined_adv * torch.clamp(ratio, 1.0 - CFG.clip_coef, 1.0 + CFG.clip_coef)
                    policy_loss = torch.max(pg1, pg2).mean()
                else:
                    reward_pg1 = -reward_adv_t[mb_idx] * ratio
                    reward_pg2 = -reward_adv_t[mb_idx] * torch.clamp(ratio, 1.0 - CFG.clip_coef, 1.0 + CFG.clip_coef)
                    reward_loss = torch.max(reward_pg1, reward_pg2).mean()
                    cost_surrogate = (ratio * cost_adv_t[mb_idx]).mean() + (mean_cost - CFG.cost_limit)
                    safety_loss = CFG.cpo_safety_penalty * torch.relu(cost_surrogate)
                    policy_loss = reward_loss + safety_loss

                reward_value = reward_critic(obs_t[mb_idx]).squeeze(-1)
                cost_value = cost_critic(obs_t[mb_idx]).squeeze(-1)
                value_loss = F.mse_loss(reward_value, reward_ret_t[mb_idx])
                cost_value_loss = F.mse_loss(cost_value, cost_ret_t[mb_idx])
                loss = policy_loss + CFG.vf_coef * (value_loss + cost_value_loss) - CFG.ent_coef * entropy

                if method == "cpo_style" and float(approx_kl.item()) > 1.5 * CFG.cpo_target_kl:
                    skipped_updates += 1
                    continue

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(list(actor.parameters()) + list(reward_critic.parameters()) + list(cost_critic.parameters()), CFG.max_grad_norm)
                optimizer.step()

        if method == "lagrangian":
            lagrange_multiplier = max(0.0, lagrange_multiplier + CFG.lagrangian_lr * (mean_cost - CFG.cost_limit))

        if global_step >= CFG.adapt_timesteps:
            break

    env.close()
    info = {
        "method": method,
        "mean_rollout_cost": float(np.mean(cost_history)) if cost_history else 0.0,
        "final_lagrange_multiplier": float(lagrange_multiplier),
        "skipped_updates": int(skipped_updates),
        "global_step": int(global_step),
    }
    return actor.cpu(), reward_critic.cpu(), info

## Run Adaptation And Re-Certify


In [ ]:
def summarize_policy(
    actor: Any,
    task: DownstreamTask,
    *,
    method_name: str,
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    metrics = evaluate_actor_on_task(actor, task, episodes=CFG.eval_episodes, seed=CFG.seed + 60_000, safety_cost=True)
    safety_rows = verify_policy_safety(actor, task, label=f"{method_name}_{task.name}")
    cert_rows = [row for row in safety_rows if row.get("check") == "certificate"]
    ibp_margins = [row["margin"] for row in cert_rows if row.get("method") == "ibp"]
    crown_margins = [row["margin"] for row in cert_rows if row.get("method") in {"crown", "alpha_crown"}]
    row = {
        "task": task.name,
        "method": method_name,
        **metrics,
        "ibp_min_margin": float(min(ibp_margins)) if ibp_margins else float("nan"),
        "crown_min_margin": float(min(crown_margins)) if crown_margins else float("nan"),
        "certified": bool(cert_rows and all(row["certified"] for row in cert_rows)),
    }
    if extra:
        row.update(extra)
    return row


def error_summary(task: DownstreamTask, method: str, exc: Exception) -> dict[str, Any]:
    return {
        "task": task.name,
        "method": method,
        "mean_reward": float("nan"),
        "std_reward": float("nan"),
        "failure_rate": float("nan"),
        "success_rate": float("nan"),
        "ibp_min_margin": float("nan"),
        "crown_min_margin": float("nan"),
        "certified": False,
        "error": repr(exc),
    }


def save_policy_artifacts(actor: nn.Sequential, critic: nn.Sequential, task: DownstreamTask, method: str, summary: dict[str, Any]) -> None:
    out_dir = ARTIFACT_ROOT / task.name / method
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(actor.state_dict(), out_dir / "actor.pt")
    torch.save(critic.state_dict(), out_dir / "critic.pt")
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")


all_results: list[dict[str, Any]] = []

if RUN_FULL_PIPELINE and SOURCE_READY and source_actor is not None:
    for task_name in SELECTED_TASK_NAMES:
        task = DOWNSTREAM_TASKS[task_name]
        print(f"\n=== Downstream task: {task.name} ===")

        noadapt_summary = summarize_policy(source_actor, task, method_name="noadapt")
        all_results.append(noadapt_summary)

        try:
            rash_actor, rash_critic, rash_info = adapt_with_rashomon_pgd(source_actor, source_critic, task)
            rash_summary = summarize_policy(rash_actor, task, method_name="rashomon_pgd", extra={"adapt_info": rash_info})
            save_policy_artifacts(rash_actor, rash_critic, task, "rashomon_pgd", rash_summary)
            all_results.append(rash_summary)
        except Exception as exc:
            print(f"Rashomon-PGD skipped/failed: {exc!r}")
            all_results.append(error_summary(task, "rashomon_pgd", exc))

        try:
            lag_actor, lag_critic, lag_info = train_constrained_ppo(source_actor, source_critic, task, method="lagrangian")
            lag_summary = summarize_policy(lag_actor, task, method_name="lagrangian", extra={"adapt_info": lag_info})
            save_policy_artifacts(lag_actor, lag_critic, task, "lagrangian", lag_summary)
            all_results.append(lag_summary)
        except Exception as exc:
            print(f"Lagrangian skipped/failed: {exc!r}")
            all_results.append(error_summary(task, "lagrangian", exc))

        try:
            cpo_actor, cpo_critic, cpo_info = train_constrained_ppo(source_actor, source_critic, task, method="cpo_style")
            cpo_summary = summarize_policy(cpo_actor, task, method_name="cpo_style", extra={"adapt_info": cpo_info})
            save_policy_artifacts(cpo_actor, cpo_critic, task, "cpo_style", cpo_summary)
            all_results.append(cpo_summary)
        except Exception as exc:
            print(f"CPO-style skipped/failed: {exc!r}")
            all_results.append(error_summary(task, "cpo_style", exc))

    results_df = pd.DataFrame(all_results)
    display(results_df)
    results_path = ARTIFACT_ROOT / "results_summary.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Wrote {results_path}")
else:
    print(
        "Pipeline not run. Set RUN_FULL_PIPELINE=True and ensure the loaded source policy "
        "passes the performance and safety-certification gate."
    )

## Interpreting Results

Treat a downstream method as accepted only if:

- The sparse-reward evaluation is acceptable for the task.
- Sampled safety violations are zero or below the chosen tolerance.
- IBP and CROWN/Alpha-CROWN margins are positive over the safety box.

Rashomon-PGD is the method designed to preserve a certified finite safety set during adaptation. Lagrangian PPO and CPO-style updates use online safety costs, so they still require final certification before deployment.